# T4 / EXP-02C GPU runner (Kaggle or Colab)

Produces everything Section 3-5 of `notebooks/02_Perception_Ensemble_Tiling_EXP02.ipynb` needs, executes that notebook
on the GPU and packs the outputs into `t4_outputs.zip`. It also writes Submission #2:
`submissions/submission_exp02_perception.csv`.

**Settings.**
- **Kaggle:** open this notebook from the competition page (the dataset is mounted automatically), then set
  *Accelerator* = **GPU T4** (x1 is enough) and *Internet* = **on** (Hugging Face model downloads).
- **Colab:** *Runtime -> Change runtime type -> T4 GPU*, and put the dataset folder (the one with `scene_schema.json`)
  on Google Drive.

**Steps (each one stops the run on failure).**
1. Clone the branch and install the pinned requirements, keeping the platform's CUDA torch.
2. Check the GPU, then run the 130+ model-free unit tests.
3. MM-Grounding-DINO Large smoke test on 2 images (prints s/img: multiply by 180 to project the full pass).
4. Full MM-GDINO pass over 180 images x 5 views x 5 prompt blocks. It is **resumable**: after a disconnect, re-run
   from step 1.
5. Optional: the LVIS-alias prompt variant (`RUN_ALIAS = True`, doubles step 4).
6. EXP-01 masks with SAM2.1, for the paired Mask F1 comparison.
7. Execute notebook 02. It fits calibration, WBF and the group thresholds on CAL, segments the frozen EXP-02C boxes
   with SAM2.1 Base+, writes the submission and runs the 8-test suite.
8. Zip the outputs into `/kaggle/working/t4_outputs.zip` (or `/content/t4_outputs.zip`).

In [ ]:
BRANCH = "feat/t4-ensemble-wbf"
REPO_URL = "https://github.com/fededlV/TPI-IA.git"
RUN_ALIAS = False  # optional MM-GDINO alias variant (+1x the detection time)
COLAB_DATASET_DIR = "/content/drive/MyDrive/TPI-IA/dataset"  # Colab only: folder that contains scene_schema.json

import os, sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
WORK = Path("/content" if ON_COLAB else "/kaggle/working")
REPO = WORK / "TPI-IA"


def sh(cmd):
    """Run a shell command with live output; raise if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TPI_DATASET_DIR"] = COLAB_DATASET_DIR
if not REPO.exists():
    sh(f"git clone -b {BRANCH} --depth 1 {REPO_URL} {REPO}")
os.chdir(REPO)
sh("git log --oneline -1")
sh('grep -v "^torch" requirements.txt | pip install -q -r /dev/stdin')

In [ ]:
import torch

assert torch.cuda.is_available(), "No CUDA device: enable a T4 GPU accelerator for this session"
print(torch.__version__, torch.cuda.get_device_name(0))
sh("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv")
sh("python -m unittest discover -s tests")

In [ ]:
sh("python scripts/run_exp02_inference.py --backbone mmgdino_large --limit 2")

In [ ]:
sh("python scripts/run_exp02_inference.py --backbone mmgdino_large")
if RUN_ALIAS:
    sh("python scripts/run_exp02_inference.py --backbone mmgdino_large --prompts alias")

In [ ]:
# EXP-01 masks for the paired Mask F1 comparison (the 39 MB scene cache is git-ignored, so it is rebuilt here)
sh("python scripts/run_exp01_inference.py segment --threshold 0.175")

In [ ]:
# Sections 1-5: calibration, anti-collision WBF, group thresholds (all on CAL), VAL comparison, SAM2.1 scenes,
# Submission #2 and the 8-test suite. nbclient prints nothing until the notebook finishes.
sh("python scripts/build_notebook_02.py --execute")
sh("python -c \"from src.lvis_evaluator import run_full_verification_suite as s; "
   "print(s('submissions/submission_exp02_perception.csv'))\"")

In [ ]:
OUT = WORK / "t4_outputs.zip"
paths = ["artifacts/cache_exp02/mmgdino_large", "artifacts/cache_exp02/exp02c",
         "notebooks/02_Perception_Ensemble_Tiling_EXP02.ipynb", "submissions/submission_exp02_perception.csv"]
sh(f"rm -f {OUT} && zip -qr {OUT} " + " ".join(paths))
sh(f"cp submissions/submission_exp02_perception.csv {WORK}/")
sh(f"ls -lh {OUT} {WORK}/submission_exp02_perception.csv")
print("Download t4_outputs.zip (Output panel / Files) and unzip it at the repository root.")